In [15]:
import os
import json
import base64
from google.colab import drive
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidSignature

# Google Drive
drive.mount('/content/drive')

caminho_drive = '/content/drive/MyDrive/Segurança da Informação/Entrega 1'
arquivo_heroi = os.path.join(caminho_drive, 'heroi.png')
arquivo_script = os.path.join(caminho_drive, 'script.txt')

# Funções das chaves
def salvar_chave_privada(chave, filename):
    pem = chave.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    with open(os.path.join(caminho_drive, filename), "wb") as f:
        f.write(pem)

def salvar_chave_publica(chave, filename):
    pem = chave.public_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PublicFormat.SubjectPublicKeyInfo
    )
    with open(os.path.join(caminho_drive, filename), "wb") as f:
        f.write(pem)

def carregar_chave_privada(filename):
    with open(os.path.join(caminho_drive, filename), "rb") as f:
        return serialization.load_pem_private_key(f.read(), password=None)

def carregar_chave_publica(filename):
    with open(os.path.join(caminho_drive, filename), "rb") as f:
        return serialization.load_pem_public_key(f.read())

# Geração dos pares de chaves
diretor_priv = rsa.generate_private_key(public_exponent=65537, key_size=2048)
dev_priv = rsa.generate_private_key(public_exponent=65537, key_size=2048)
impostor_priv = rsa.generate_private_key(public_exponent=65537, key_size=2048)

# salvar chaves
salvar_chave_privada(diretor_priv, "diretor_priv.pem")
salvar_chave_publica(diretor_priv.public_key(), "diretor_pub.pem")

salvar_chave_privada(dev_priv, "dev_priv.pem")
salvar_chave_publica(dev_priv.public_key(), "dev_pub.pem")

salvar_chave_privada(impostor_priv, "impostor_priv.pem")
salvar_chave_publica(impostor_priv.public_key(), "impostor_pub.pem")

print("par de chaves exportado")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
par de chaves exportado


In [16]:
# Leitura dos arquivos
with open(arquivo_heroi, "rb") as f:
    pixelart_bytes = f.read()

with open(arquivo_script, "r", encoding="utf-8") as f:
    script_texto = f.read()

# Empacotamento
payload_bruto = {
    "nome_imagem": "heroi_pixelart.png",
    "pixelart_b64": base64.b64encode(pixelart_bytes).decode('ascii'),
    "nome_script": "script_ideia.txt",
    "script_texto": script_texto
}
payload_bytes = json.dumps(payload_bruto).encode('utf-8')

# Cifragem simétrica
chave_aes = AESGCM.generate_key(bit_length=256)
aesgcm = AESGCM(chave_aes)
nonce = os.urandom(12)
payload_cifrado = aesgcm.encrypt(nonce, payload_bytes, None)

# Cifragem assimétrica da chave com a chave pública do Dev
chave_aes_cifrada = dev_priv.public_key().encrypt(
    chave_aes,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Assinatura digital com a chave privada do Diretor
assinatura = diretor_priv.sign(
    payload_cifrado,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

# Montagem
pacote_final = {
    "payload_cifrado": base64.b64encode(payload_cifrado).decode('ascii'),
    "chave_aes_cifrada": base64.b64encode(chave_aes_cifrada).decode('ascii'),
    "nonce": base64.b64encode(nonce).decode('ascii'),
    "assinatura": base64.b64encode(assinatura).decode('ascii')
}

caminho_pacote_json = os.path.join(caminho_drive, "pacote_cifrado.json")
with open(caminho_pacote_json, "w") as f:
    json.dump(pacote_final, f, indent=4)

print("Pacote cifrado e assinado")

Pacote cifrado e assinado


In [17]:
try:
    with open(os.path.join(caminho_drive, "pacote_cifrado.json"), "r") as f:
        pacote = json.load(f)

    payload_cifrado = base64.b64decode(pacote["payload_cifrado"])
    chave_aes_cifrada = base64.b64decode(pacote["chave_aes_cifrada"])
    nonce = base64.b64decode(pacote["nonce"])
    assinatura = base64.b64decode(pacote["assinatura"])

    dev_priv = carregar_chave_privada("dev_priv.pem")
    diretor_pub = carregar_chave_publica("diretor_pub.pem")

    # Validação da assinatura do Diretor
    diretor_pub.verify(
        assinatura,
        payload_cifrado,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )

    # Decifragem da chave AES com a chave privada do Dev
    chave_aes = dev_priv.decrypt(
        chave_aes_cifrada,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

    # Decifragem do payload
    aesgcm = AESGCM(chave_aes)
    payload_bytes = aesgcm.decrypt(nonce, payload_cifrado, None)

    # Desempacotamento e reconstrução
    payload_dict = json.loads(payload_bytes.decode('utf-8'))

    caminho_png_saida = os.path.join(caminho_drive, "heroi_restaurado.png")
    caminho_txt_saida = os.path.join(caminho_drive, "script_restaurado.txt")

    pixelart_bytes_restaurados = base64.b64decode(payload_dict["pixelart_b64"])
    with open(caminho_png_saida, "wb") as f:
        f.write(pixelart_bytes_restaurados)

    with open(caminho_txt_saida, "w", encoding="utf-8") as f:
        f.write(payload_dict["script_texto"])

    print("Assinatura validada e arquivos restaurados no Drive.")
    print(f"imagem: {caminho_png_saida}")
    print(f"script: {caminho_txt_saida}\n")
except Exception as e:
    print(f"erro ao processar o pacote: {e}")

Assinatura validada e arquivos restaurados no Drive.
imagem: /content/drive/MyDrive/Segurança da Informação/Entrega 1/heroi_restaurado.png
script: /content/drive/MyDrive/Segurança da Informação/Entrega 1/script_restaurado.txt



In [18]:
# teste de falha com chave privada incorreta
try:
    with open(os.path.join(caminho_drive, "pacote_cifrado.json"), "r") as f:
        pacote = json.load(f)

    chave_aes_cifrada = base64.b64decode(pacote["chave_aes_cifrada"])
    impostor_priv = carregar_chave_privada("impostor_priv.pem")

    impostor_priv.decrypt(
        chave_aes_cifrada,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    print("falha de segurança, chave incorreta decifrou o pacote")

except Exception as e:
    print("falha na decifragem ")
    print(f"Exceção capturada: {type(e).__name__}")

falha na decifragem 
Exceção capturada: ValueError


In [21]:
# teste de falha com validação de assinatura com chave incorreta
try:
    with open(os.path.join(caminho_drive, "pacote_cifrado.json"), "r") as f:
        pacote = json.load(f)

    payload_cifrado = base64.b64decode(pacote["payload_cifrado"])
    assinatura = base64.b64decode(pacote["assinatura"])
    impostor_pub = carregar_chave_publica("impostor_pub.pem")

    impostor_pub.verify(
        assinatura,
        payload_cifrado,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("falha, assinatura inválida foi aceita")

except Exception as e:
    print("Assinatura recusada com sucesso.")
    print(f"Exceção capturada: {type(e).__name__}")

Assinatura recusada com sucesso.
Exceção capturada: InvalidSignature
